# TFT Model Evaluation (10-Second Bars)

This notebook evaluates the performance of a Temporal Fusion Transformer model trained on 10-second bar data for cryptocurrency price prediction.

In [1]:
import os
import glob
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from datetime import datetime, timedelta
from models.temporal_fusion_transformer import TemporalFusionTransformer, MarketDataset
from torch.utils.data import DataLoader

# Define a custom dataset for NumPy arrays
class NumpyMarketDataset(torch.utils.data.Dataset):
    def __init__(self, features, targets, context_length=10):
        """A simple dataset for handling NumPy feature arrays for the TFT model.
        
        Args:
            features: NumPy array of shape (n_samples, n_features)
            targets: NumPy array of shape (n_samples,)
            context_length: Length of context window
        """
        self.features = features
        self.targets = targets
        self.context_length = context_length
        
        # Calculate valid samples (need enough history for context window)
        self.n_samples = max(0, len(features) - context_length)
        
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        # Get window of data (idx to idx+context_length)
        feature_window = self.features[idx:idx+self.context_length]
        # Get target (the next value after the window)
        target = self.targets[idx+self.context_length-1]  # Use the last value in window as target
        
        # Convert to tensors
        feature_tensor = torch.tensor(feature_window, dtype=torch.float32)
        target_tensor = torch.tensor(target, dtype=torch.float32).reshape(1)
        
        return feature_tensor, target_tensor

# Configure plotting
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 8)

## Database Connection

Connect to the database to retrieve 10-second bar data.

In [2]:
# Database configuration
DB_CONFIG = {
    'host': 'localhost',
    'port': 5438,
    'user': 'backtest_user',
    'password': 'backtest_password',
    'database': 'backtest_db'
}

# Connect to the database
conn = psycopg2.connect(
    host=DB_CONFIG['host'],
    port=DB_CONFIG['port'],
    user=DB_CONFIG['user'],
    password=DB_CONFIG['password'],
    database=DB_CONFIG['database']
)

print("Database connection established")

Database connection established


## Data Loading

Retrieve 10-second bar data from the database.

In [3]:
# Query to get test data
query = """
SELECT 
    timestamp,
    open, high, low, close,
    volume, bid_vol, ask_vol, imbalance, 
    spread_mean, spread_pct_mean,
    new_bid_orders, new_ask_orders,
    canceled_bid_orders, canceled_ask_orders,
    executed_bid_orders, executed_ask_orders,
    buy_sell_imbalance,
    returns_10sec, returns_30sec, returns_1min,
    volatility_1min, target_10sec as next_return
FROM tft_features_10sec
ORDER BY timestamp DESC
LIMIT 10000;
"""

# Load data
df = pd.read_sql_query(query, conn)
print(f"Loaded {len(df)} rows of 10-second bar data")
df = df.sort_values('timestamp')
df.head()

/var/folders/nl/1k350bws5sdcymcpn81f46j40000gn/T/ipykernel_46643/3071912364.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


Loaded 10000 rows of 10-second bar data


,timestamp,open,high,low,close,volume,bid_vol,ask_vol,imbalance,spread_mean,...,canceled_bid_orders,canceled_ask_orders,executed_bid_orders,executed_ask_orders,buy_sell_imbalance,returns_10sec,returns_30sec,returns_1min,volatility_1min,next_return
9999,2025-04-29 20:11:50+00:00,0.83665,0.83665,0.83665,0.83665,None,None,None,-0.951484,None,...,671,672,None,None,None,0.0,0.0,0.0,0.0,0.0
9998,2025-04-29 20:12:00+00:00,0.83665,0.83665,0.83665,0.83665,None,None,None,-0.951484,None,...,0,0,None,None,None,0.0,0.0,0.0,0.0,0.0
9997,2025-04-29 20:12:10+00:00,0.83665,0.83665,0.83665,0.83665,None,None,None,-0.951484,None,...,0,0,None,None,None,0.0,0.0,0.0,0.0,0.0
9996,2025-04-29 20:12:20+00:00,0.83665,0.83665,0.83665,0.83665,None,None,None,-0.951484,None,...,0,0,None,None,None,0.0,0.0,0.0,0.0,0.0
9995,2025-04-29 20:12:30+00:00,0.83665,0.83665,0.83665,0.83665,None,None,None,-0.951484,None,...,0,0,None,None,None,0.0,0.0,0.0,0.0,0.0


## Data Preparation

Prepare the data for model evaluation.

In [4]:
# Define feature columns
feature_columns = [col for col in df.columns if col not in ['timestamp', 'next_return']]
print(f"Using {len(feature_columns)} features")

# Normalize features
df_scaled = df.copy()
for col in feature_columns:
    mean_val = df[col].mean()
    std_val = df[col].std()
    
    if std_val == 0:
        print(f"Feature {col} has zero std dev, setting to zero")
        df_scaled[col] = 0
    else:
        df_scaled[col] = (df[col] - mean_val) / std_val

# Fill NaN values with zeros
df_scaled = df_scaled.fillna(0)
df_scaled['target'] = df_scaled['next_return']

Using 21 features
Feature returns_10sec has zero std dev, setting to zero
Feature returns_30sec has zero std dev, setting to zero
Feature returns_1min has zero std dev, setting to zero
Feature volatility_1min has zero std dev, setting to zero


## Model Loading

Load the TFT model (with bias correction if available).

In [5]:
# Check for bias-corrected models first
bias_corrected_models = [f for f in os.listdir('models') 
                         if f.startswith('tft_10sec_bias_corrected_') and f.endswith('.pt')]

if bias_corrected_models:
    # Sort by timestamp (newest first)
    bias_corrected_models.sort(reverse=True)
    model_path = os.path.join('models', bias_corrected_models[0])
    print(f"Loading bias-corrected model: {model_path}")
    
    # Load the model data
    loaded_data = torch.load(model_path)
    print(f"Loaded data type: {type(loaded_data)}")
    
    # Initialize a new model
    model = TemporalFusionTransformer(
        time_varying_features=21,  # Use exactly 21 features
        hidden_size=128,  # IMPORTANT: Match the saved model architecture
        num_heads=4,
        dropout=0.2,
        learning_rate=1e-4,
        context_length=30,
        prediction_length=1,
        bias_correction=True
    )
    
    # Get bias correction value (default 0.0002 if not found)
    bias_correction = loaded_data.get('bias_correction', 0.0002)
    print(f"Using bias correction value: {bias_correction}")
    
    # Check the type of model data and load appropriately
    if 'model' in loaded_data:
        model_data = loaded_data['model']
        print(f"Model data type: {type(model_data)}")
        
        if isinstance(model_data, torch.nn.Module):
            # It's already a model instance, use it directly
            model = model_data
            print("Using loaded model instance directly")
        elif isinstance(model_data, dict) or isinstance(model_data, collections.OrderedDict):
            # Custom loading for mismatched architectures
            print("Using custom parameter loading for different model architectures")
            
            # Get current model state dict
            current_state = model.state_dict()
            
            # Copy compatible parameters
            compatible_params = 0
            for param_name, param in current_state.items():
                if param_name in model_data and param.shape == model_data[param_name].shape:
                    # Copy parameter if name and shape match
                    current_state[param_name] = model_data[param_name]
                    compatible_params += 1
                    print(f"Loaded compatible parameter: {param_name}")
            
            # Load the updated state dict
            model.load_state_dict(current_state, strict=False)
            print(f"Loaded {compatible_params} compatible parameters from state dict")
            
            # Now check if the feature_layer and final_layer were loaded
            critical_layers = ['feature_layer.weight', 'feature_layer.bias', 'final_layer.0.weight', 'final_layer.0.bias']
            for layer in critical_layers:
                if layer in model_data and layer in current_state:
                    print(f"Critical layer '{layer}' was loaded correctly")
                else:
                    print(f"WARNING: Critical layer '{layer}' could not be loaded")
        else:
            print(f"WARNING: Unknown model data type {type(model_data)}, using new model")
    else:
        # No 'model' key, try loading the whole thing as a state dict
        try:
            current_state = model.state_dict()
            compatible_params = 0
            
            for param_name, param in current_state.items():
                if param_name in loaded_data and param.shape == loaded_data[param_name].shape:
                    current_state[param_name] = loaded_data[param_name]
                    compatible_params += 1
            
            model.load_state_dict(current_state, strict=False)
            print(f"Loaded {compatible_params} compatible parameters from full data")
        except Exception as e:
            print(f"WARNING: Failed to load model data: {str(e)}")
    
    # Put model in evaluation mode
    model.eval()
    print(f"Model loaded with bias correction: {bias_correction}")
    
    # Create a wrapper for bias correction
    def predict_with_bias_correction(x):
        with torch.no_grad():
            raw_prediction = model(None, x)  # Pass None for static_features
            return raw_prediction - bias_correction
else:
    # Try regular model files
    regular_models = [f for f in os.listdir('models') 
                      if f.startswith('tft_10sec_model_') and f.endswith('.pt')]
    
    if regular_models:
        regular_models.sort(reverse=True)
        model_path = os.path.join('models', regular_models[0])
        print(f"Loading regular model: {model_path}")
        
        # Load the data
        loaded_data = torch.load(model_path)
        print(f"Loaded data type: {type(loaded_data)}")
        
        # Initialize a new model
        model = TemporalFusionTransformer(
            time_varying_features=21,  # Use exactly 21 features
            hidden_size=128,  # IMPORTANT: Match the saved model architecture
            num_heads=4,
            dropout=0.2,
            learning_rate=1e-4,
            context_length=30,
            prediction_length=1
        )
        
        # Handle different formats with custom loading
        if hasattr(loaded_data, 'state_dict'):
            # It's a model object
            model = loaded_data
            print("Using loaded model instance directly")
        else:
            # Custom parameter loading for state dict
            print("Using custom parameter loading for different model architectures")
            current_state = model.state_dict()
            compatible_params = 0
            
            for param_name, param in current_state.items():
                if param_name in loaded_data and param.shape == loaded_data[param_name].shape:
                    current_state[param_name] = loaded_data[param_name]
                    compatible_params += 1
            
            model.load_state_dict(current_state, strict=False)
            print(f"Loaded {compatible_params} compatible parameters from state dict")
        
        # Put model in evaluation mode
        model.eval()
        
        # No bias correction
        def predict_with_bias_correction(x):
            with torch.no_grad():
                return model(None, x)  # Pass None for static_features
    else:
        print("No 10-second TFT models found")
        raise FileNotFoundError("No models found")

Loading bias-corrected model: models/tft_10sec_bias_corrected_20250501_230054.pt
Loaded data type: <class 'dict'>
Using bias correction value: 0.0002
Model data type: <class 'collections.OrderedDict'>
Using custom parameter loading for different model architectures
Loaded compatible parameter: feature_layer.weight
Loaded compatible parameter: feature_layer.bias
Loaded compatible parameter: temporal_processing.weight_ih_l0
Loaded compatible parameter: temporal_processing.weight_hh_l0
Loaded compatible parameter: temporal_processing.bias_ih_l0
Loaded compatible parameter: temporal_processing.bias_hh_l0
Loaded compatible parameter: temporal_processing.weight_ih_l1
Loaded compatible parameter: temporal_processing.weight_hh_l1
Loaded compatible parameter: temporal_processing.bias_ih_l1
Loaded compatible parameter: temporal_processing.bias_hh_l1
Loaded compatible parameter: self_attention.in_proj_weight
Loaded compatible parameter: self_attention.in_proj_bias
Loaded compatible parameter: sel

## Creating Data Loaders

In [6]:
# Prepare data for the model
X = df_scaled[feature_columns].values.astype(np.float32)
y = df_scaled['target'].values.astype(np.float32)

# Create dataset and dataloader
context_length = 30
dataset = NumpyMarketDataset(X, y, context_length=context_length)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False)

print(f"Created dataset with {len(dataset)} samples")

Created dataset with 9970 samples


## Generating Predictions

In [7]:
# Generate predictions
actuals = []
predictions = []
timestamps = []

# Skip first context_length points (they don't have enough history)
valid_timestamps = df['timestamp'].values[context_length:]

for i, (X_batch, y_batch) in enumerate(dataloader):
    # Check for feature dimension mismatch
    if X_batch.shape[2] != len(feature_columns):
        print(f"WARNING: Feature dimension mismatch. Expected {len(feature_columns)}, got {X_batch.shape[2]}")
        
        # Handle mismatch by using only the first n features if needed
        if X_batch.shape[2] > len(feature_columns):
            X_batch = X_batch[:, :, :len(feature_columns)]
    
    # Make predictions with bias correction if available
    pred_batch = predict_with_bias_correction(X_batch)
    
    # Store results
    actuals.extend(y_batch.numpy().flatten())
    predictions.extend(pred_batch.numpy().flatten())
    
    # Add timestamps
    batch_size = len(y_batch)
    start_idx = i * dataloader.batch_size
    end_idx = min(start_idx + batch_size, len(valid_timestamps))
    timestamps.extend(valid_timestamps[start_idx:end_idx])

# Create results dataframe
results_df = pd.DataFrame({
    'timestamp': timestamps,
    'actual': actuals,
    'predicted': predictions
})

# Calculate metrics
mae = np.mean(np.abs(results_df['actual'] - results_df['predicted']))
rmse = np.sqrt(np.mean((results_df['actual'] - results_df['predicted'])**2))
correlation = np.corrcoef(results_df['actual'], results_df['predicted'])[0, 1]

print(f"Evaluation metrics:")
print(f"MAE: {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"Correlation: {correlation:.6f}")

# Check for bias in predictions
bias = np.mean(results_df['predicted'] - results_df['actual'])
print(f"Prediction bias: {bias:.6f}")

results_df.head()

Evaluation metrics:
MAE: 0.309757
RMSE: 0.323717
Correlation: nan
Prediction bias: -0.309655


/Users/derekburns/anaconda3/lib/python3.11/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/derekburns/anaconda3/lib/python3.11/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,timestamp,actual,predicted
0,2025-04-29 20:16:50,0.0,-0.419942
1,2025-04-29 20:17:00,0.0,-0.405503
2,2025-04-29 20:17:10,0.0,-0.388760
3,2025-04-29 20:17:20,0.0,-0.387243
4,2025-04-29 20:17:30,0.0,-0.384359


## Visualizations

In [ ]:
# Plot actual vs predicted values
plt.figure(figsize=(14, 7))
plt.plot(results_df['timestamp'], results_df['actual'], label='Actual')
plt.plot(results_df['timestamp'], results_df['predicted'], label='Predicted')
plt.title('Actual vs Predicted Returns (10-second bars)')
plt.xlabel('Time')
plt.ylabel('Return')
plt.legend()
plt.tight_layout()
plt.show()

# Zoomed view (100 points)
window_size = 100
start_idx = len(results_df) // 2
window_df = results_df.iloc[start_idx:start_idx+window_size]

plt.figure(figsize=(14, 7))
plt.plot(window_df['timestamp'], window_df['actual'], label='Actual')
plt.plot(window_df['timestamp'], window_df['predicted'], label='Predicted')
plt.title('Actual vs Predicted Returns (Zoomed View)')
plt.xlabel('Time')
plt.ylabel('Return')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Error distribution
plt.figure(figsize=(10, 6))
errors = results_df['predicted'] - results_df['actual']
sns.histplot(errors, kde=True)
plt.axvline(x=0, color='r', linestyle='--')
plt.title('Prediction Error Distribution')
plt.xlabel('Error')
plt.tight_layout()
plt.show()

## Trading Simulation

Simulate a simple trading strategy based on the model's predictions.

In [ ]:
# Simple trading strategy
def simulate_trading(df, threshold=0.0001, fee=0.001):
    """Simple trading simulation"""
    position = 0  # 0=none, 1=long, -1=short
    trades = []
    
    for i, row in df.iterrows():
        pred = row['predicted']
        
        # Trading logic
        if position == 0:  # No position
            if pred > threshold:
                position = 1
                trades.append({'timestamp': row['timestamp'], 'action': 'buy', 
                              'pred': pred, 'actual': row['actual']})
            elif pred < -threshold:
                position = -1
                trades.append({'timestamp': row['timestamp'], 'action': 'sell', 
                              'pred': pred, 'actual': row['actual']})
        
        elif position == 1:  # Long position
            if pred < 0:
                position = 0
                trades.append({'timestamp': row['timestamp'], 'action': 'close_long', 
                              'pred': pred, 'actual': row['actual']})
        
        elif position == -1:  # Short position
            if pred > 0:
                position = 0
                trades.append({'timestamp': row['timestamp'], 'action': 'close_short', 
                              'pred': pred, 'actual': row['actual']})
    
    if not trades:
        print("No trades generated")
        return None
        
    # Create trades dataframe
    trades_df = pd.DataFrame(trades)
    
    # Calculate PnL
    pnl = []
    cumulative = 0
    
    for i, trade in enumerate(trades_df.iterrows()):
        if i == 0 or trades_df.iloc[i-1]['action'] in ['close_long', 'close_short']:
            # Opening trade, no PnL yet
            pnl.append(0)
        else:
            # Closing trade
            prev_action = trades_df.iloc[i-1]['action']
            direction = 1 if prev_action == 'buy' else -1
            p = direction * trade[1]['actual'] - fee
            pnl.append(p)
            cumulative += p
    
    trades_df['pnl'] = pnl
    trades_df['cumulative_pnl'] = np.cumsum(pnl)
    
    # Print statistics
    win_count = (trades_df['pnl'] > 0).sum()
    total_trades = len(trades_df)
    win_rate = win_count / total_trades if total_trades > 0 else 0
    
    print(f"Total trades: {total_trades}")
    print(f"Win rate: {win_rate:.2%}")
    print(f"Total PnL: {cumulative:.6f}")
    
    return trades_df

# Run simulation with different thresholds
for threshold in [0.0001, 0.0002, 0.0003]:
    print(f"\nSimulation with threshold {threshold}:")
    trades = simulate_trading(results_df, threshold=threshold)
    
    if trades is not None and len(trades) > 0:
        plt.figure(figsize=(10, 6))
        plt.plot(trades.index, trades['cumulative_pnl'])
        plt.title(f'Cumulative PnL (threshold={threshold})')
        plt.xlabel('Trade')
        plt.ylabel('Cumulative PnL')
        plt.grid(True)
        plt.tight_layout()
        plt.show()

## Conclusion

The 10-second TFT model allows for high-frequency prediction of price movements, capturing more short-term trading opportunities compared to models with longer timeframes.

Key findings:
- [Summarize performance metrics]
- [Note any bias correction improvements]
- [Trading strategy effectiveness]

Next steps:
1. Further optimize model parameters for high-frequency data
2. Test more sophisticated trading strategies
3. Compare with 1-minute model to quantify the advantages